In [1]:
import re

from dotenv import find_dotenv, load_dotenv
from langchain.document_loaders import WebBaseLoader
from langchain.prompts import ChatPromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv(find_dotenv("../../creds/.env"), verbose=True);

In [3]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
    num_ctx=16384,  # 40960 max
).bind(think=False)

## Tagging

In [4]:
class Tagging(BaseModel):
    """Tag the piece of text with particular info."""

    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, or `neutral`")
    language: str = Field(description="language of text (should be ISO 639-1 code)")

In [5]:
tagging_functions = [convert_to_openai_function(Tagging)]
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Think carefully, and then tag the text as instructed"),
        ("user", "{input}"),
    ]
)

In [6]:
llm_with_tools = llm.bind_tools(
    tools=tagging_functions,
)

In [7]:
def extract_tool_calls(message):
    return message.tool_calls


ToolCallsExtractingRunnableLambda = RunnableLambda(extract_tool_calls)

In [8]:
tagging_chain = prompt | llm_with_tools | ToolCallsExtractingRunnableLambda

In [9]:
tagging_chain.invoke({"input": "I don't love langchain"})

[{'name': 'Tagging',
  'args': {'language': 'en', 'sentiment': 'neg'},
  'id': '90deffaf-8006-4325-80a7-68ce4047f476',
  'type': 'tool_call'}]

In [10]:
tagging_chain.invoke({"input": "Я люблю Київ"})

[{'name': 'Tagging',
  'args': {'language': 'uk', 'sentiment': 'pos'},
  'id': 'e40a994c-26b4-4b24-981a-b6c36d705b9b',
  'type': 'tool_call'}]

## Extraction

In [11]:
class Person(BaseModel):
    """Information about a person."""

    name: str = Field(description="person's name")
    age: int | None = Field(description="person's age")
    sex: str = Field(description="person's sex can be either 'm' if male or 'f' if female")


class Information(BaseModel):
    """Information to extract."""

    people: list[Person] = Field(description="List of info about people")

In [12]:
extraction_functions = [convert_to_openai_function(Information)]
extraction_model = llm.bind_tools(tools=extraction_functions)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Extract the relevant information, if not explicitly provided - do not guess. Extract partial information",
        ),
        ("human", "{input}"),
    ]
)

extraction_chain = prompt | extraction_model | ToolCallsExtractingRunnableLambda

In [13]:
extraction_chain.invoke({"input": "Joe is 30, Jess is 28 and their children - Don and April, both 7"})

[{'name': 'Information',
  'args': {'people': [{'age': 30, 'name': 'Joe', 'sex': 'm'},
    {'age': 28, 'name': 'Jess', 'sex': 'f'},
    {'age': 7, 'name': 'Don', 'sex': 'm'},
    {'age': 7, 'name': 'April', 'sex': 'f'}]},
  'id': '534fcb1e-8f81-4e0d-82e9-cc2efef3c568',
  'type': 'tool_call'}]

### Doing it for real

In [14]:
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
documents = loader.load()

In [15]:
doc = documents[0]

In [16]:
class Overview(BaseModel):
    """Overview of a section of text."""

    summary: str = Field(description="Provide a concise summary of the content.", min_length=600)
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")

In [17]:
tagging_functions = [convert_to_openai_function(Overview)]
tagging_model = llm.bind_tools(tools=tagging_functions)
tagging_chain = prompt | tagging_model | ToolCallsExtractingRunnableLambda

In [18]:
tagging_result = tagging_chain.invoke({"input": doc.page_content})
tagging_result

[{'name': 'Overview',
  'args': {'keywords': 'LLM agents, Planning, Memory, Tool Use, Chain of Thought, Tree of Thoughts, ReAct, Reflexion, Chain of Hindsight, MIPS, MRKL, TALM, Toolformer, HuggingGPT, ChemCrow, Generative Agents, Challenges',
   'language': 'English',
   'summary': 'The article explores LLM-powered autonomous agents, detailing their core components: Planning (including Chain of Thought, Tree of Thoughts, ReAct, Reflexion, and Chain of Hindsight), Memory (short-term, long-term, and Maximum Inner Product Search algorithms), and Tool Use (MRKL, TALM, Toolformer, HuggingGPT). It highlights case studies like ChemCrow and Generative Agents, discusses challenges such as finite context length and natural language interface reliability, and references key research papers.'},
  'id': 'a02200dc-1ff6-4d9f-984b-c7d50bc948f1',
  'type': 'tool_call'}]

In [19]:
class Paper(BaseModel):
    """Information about papers mentioned."""

    title: str
    author: str | None
    url: str


class Info(BaseModel):
    """Information to extract"""

    papers: list[Paper]

In [20]:
template = """
An article will be passed to you. Extract information about all the papers that are mentioned in the article.
Do not extract the name of the article itself.
If no papers are mentioned that's fine - you don't need to extract any! Just return an empty list.
Do not make up or guess ANY extra information. Only extract what exactly is in the text.
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", template),
        ("human", "{input}"),
    ]
)

extraction_functions = [convert_to_openai_function(Info)]
extraction_model = llm.bind_tools(tools=extraction_functions)
extraction_chain = prompt | extraction_model | ToolCallsExtractingRunnableLambda

In [21]:
extraction_result = extraction_chain.invoke(
    {
        "input": re.sub(r"\n+|\s+", " ", doc.page_content.strip()),
    }
)
extraction_result

[{'name': 'Info',
  'args': {'papers': [{'author': 'Wei et al.',
     'title': 'Chain of thought prompting elicits reasoning in large language models',
     'url': 'https://arxiv.org/abs/2210.07433'},
    {'author': 'Yao et al.',
     'title': 'Tree of Thoughts: Dliberate Problem Solving with Large Language Models',
     'url': 'https://arxiv.org/abs/2305.10601'},
    {'author': 'Liu et al.',
     'title': 'Chain of Hindsight Aligns Language Models with Feedback',
     'url': 'https://arxiv.org/abs/2302.02676'},
    {'author': 'Liu et al.',
     'title': 'LLM+P: Empowering Large Language Models with Optimal Planning Proficiency',
     'url': 'https://arxiv.org/abs/2304.11477'},
    {'author': 'Yao et al.',
     'title': 'ReAct: Synergizing reasoning and acting in language models',
     'url': 'https://arxiv.org/abs/2304.02514'},
    {'author': 'Shinn & Labash',
     'title': 'Reflexion: an autonomous agent with dynamic memory and self-reflection',
     'url': 'https://arxiv.org/abs/230

In [22]:
parallel_chains = RunnableParallel(
    info=extraction_chain,
    overview=tagging_chain,
)

In [23]:
final_result = parallel_chains.invoke(
    {
        "input": re.sub(r"\n+|\s+", " ", doc.page_content.strip()),
    }
)

In [24]:
final_result

{'info': [{'name': 'Info',
   'args': {'papers': [{'author': 'Wei et al.',
      'title': 'Chain of thought prompting elicits reasoning in large language models',
      'url': 'https://arxiv.org/abs/2210.07433'},
     {'author': 'Yao et al.',
      'title': 'Tree of Thoughts: Dliberate Problem Solving with Large Language Models',
      'url': 'https://arxiv.org/abs/2305.10601'},
     {'author': 'Liu et al.',
      'title': 'Chain of Hindsight Aligns Language Models with Feedback',
      'url': 'https://arxiv.org/abs/2302.02676'},
     {'author': 'Liu et al.',
      'title': 'LLM+P: Empowering Large Language Models with Optimal Planning Proficiency',
      'url': 'https://arxiv.org/abs/2304.11477'},
     {'author': 'Yao et al.',
      'title': 'ReAct: Synergizing reasoning and acting in language models',
      'url': 'https://arxiv.org/abs/2304.02514'},
     {'author': 'Shinn & Labash',
      'title': 'Reflexion: an autonomous agent with dynamic memory and self-reflection',
      'url':

In [25]:
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)
splits = text_splitter.split_text(re.sub(r"\n+|\s+", " ", doc.page_content.strip()))
len(splits)

11

In [26]:
def flatten(list_of_lists: list[list]) -> list:
    return [item for ls in list_of_lists for item in ls]


splits_preparator = RunnableLambda(
    lambda x: [{"input": doc.strip()} for doc in text_splitter.split_text(re.sub(r"\n+|\s+", " ", x.strip()))]
)

In [27]:
chain = splits_preparator | extraction_chain.map() | flatten

In [ ]:
chain.invoke(doc.page_content)